In [ ]:
import pandas as pd
import numpy as np

train_ratings = pd.read_csv(
    "../datasets/processed/train_ratings.csv"
)

test_ratings = pd.read_csv(
    "../datasets/processed/test_ratings.csv"
)

recipes = pd.read_csv(
    "../../datasets/RAW_recipes.csv"
)

print("Train shape:", train_ratings.shape)
print("Test shape:", test_ratings.shape)
print("Recipes shape:", recipes.shape)

In [ ]:
content_columns = [
    "id",
    "name",
    "tags",
    "description",
    "ingredients"
]

recipes[content_columns].head()

In [ ]:
# Popunimo eventualno nedostajuće vrednosti
recipes["tags"] = recipes["tags"].fillna("")
recipes["description"] = recipes["description"].fillna("")
recipes["ingredients"] = recipes["ingredients"].fillna("")

# Spojimo sadržaj recepta u jedan tekst
recipes["content"] = (
        recipes["tags"] + " " +
        recipes["ingredients"] + " " +
        recipes["description"]
)

print("Number of recipes:", len(recipes))
print("Example content:")
print(recipes.loc[0, "content"])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50000
)

tfidf_matrix = tfidf.fit_transform(recipes["content"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of stored values:", tfidf_matrix.nnz)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Izaberi jedan recept za testiranje
test_recipe_index = 0

# Sličnost ovog recepta sa svim ostalim receptima
similarities = cosine_similarity(
    tfidf_matrix[test_recipe_index],
    tfidf_matrix
).flatten()

# Izbaci sam recept
similarities[test_recipe_index] = -1

# Indeksi 10 najsličnijih recepata
similar_recipe_indices = np.argsort(similarities)[-10:][::-1]

# Prikaži rezultate
similar_recipes = recipes.iloc[similar_recipe_indices][
    ["id", "name"]
].copy()

similar_recipes["similarity"] = similarities[similar_recipe_indices]

similar_recipes

In [ ]:
test_user = test_ratings["user_id"].iloc[0]

user_train_ratings = train_ratings[
    train_ratings["user_id"] == test_user
    ].copy()

print("Test user:", test_user)
print("Number of training ratings:", len(user_train_ratings))

user_train_ratings

In [ ]:
# Recepti koje je korisnik ocenio
rated_recipe_ids = user_train_ratings["recipe_id"].values

# Pronađi njihove indekse u recipes tabeli
recipe_indices = recipes.index[
    recipes["id"].isin(rated_recipe_ids)
]

# TF-IDF vektori ocenjenih recepata
user_recipe_vectors = tfidf_matrix[recipe_indices]

# Ocene korisnika za te recepte
user_ratings = user_train_ratings[
    user_train_ratings["recipe_id"].isin(
        recipes.iloc[recipe_indices]["id"]
    )
]["rating"].values

print("Rated recipes found:", user_recipe_vectors.shape[0])
print("Ratings used:", len(user_ratings))

In [ ]:
# Pretvaramo ocene u težine
weights = user_ratings.astype(float)

# Ponderisani profil korisnika
user_profile = user_recipe_vectors.multiply(
    weights.reshape(-1, 1)
).sum(axis=0)

# Normalizujemo profil
user_profile = user_profile / weights.sum()

print("User profile shape:", user_profile.shape)

In [ ]:
# Pretvaramo user profile iz np.matrix u običan numpy array
user_profile = np.asarray(user_profile)

# Proveravamo oblik
print("User profile shape:", user_profile.shape)

# Sličnost user profila sa svim receptima
user_similarities = cosine_similarity(
    user_profile,
    tfidf_matrix
).flatten()

# Ne preporučujemo recepte koje je korisnik već ocenio
rated_indices = recipes.index[
    recipes["id"].isin(rated_recipe_ids)
]

user_similarities[rated_indices] = -1

# Top 10 preporuka
top_indices = np.argsort(user_similarities)[-10:][::-1]

content_recommendations = recipes.iloc[top_indices][
    ["id", "name"]
].copy()

content_recommendations["score"] = user_similarities[top_indices]

content_recommendations

In [ ]:
def recommend_content_based(user_id, k=10):
    # Ocene korisnika iz trening skupa
    user_ratings = train_ratings[
        train_ratings["user_id"] == user_id
        ].copy()

    if len(user_ratings) == 0:
        return pd.DataFrame(columns=["id", "name", "score"])

    # Recepti koje je korisnik ocenio
    rated_recipe_ids = user_ratings["recipe_id"].values

    # Pronađi njihove indekse u recipes tabeli
    recipe_indices = recipes.index[
        recipes["id"].isin(rated_recipe_ids)
    ]

    if len(recipe_indices) == 0:
        return pd.DataFrame(columns=["id", "name", "score"])

    # TF-IDF vektori ocenjenih recepata
    user_recipe_vectors = tfidf_matrix[recipe_indices]

    # Ocene korisnika
    valid_ratings = user_ratings[
        user_ratings["recipe_id"].isin(
            recipes.iloc[recipe_indices]["id"]
        )
    ]["rating"].values.astype(float)

    # User profile
    profile = user_recipe_vectors.multiply(
        valid_ratings.reshape(-1, 1)
    ).sum(axis=0)

    profile = profile / valid_ratings.sum()
    profile = np.asarray(profile)

    # Sličnost sa svim receptima
    similarities = cosine_similarity(
        profile,
        tfidf_matrix
    ).flatten()

    # Ne preporučujemo već ocenjene recepte
    rated_indices = recipes.index[
        recipes["id"].isin(rated_recipe_ids)
    ]

    similarities[rated_indices] = -1

    # Top-k
    top_indices = np.argsort(similarities)[-k:][::-1]

    recommendations = recipes.iloc[top_indices][
        ["id", "name"]
    ].copy()

    recommendations["score"] = similarities[top_indices]

    return recommendations

In [ ]:
recommend_content_based(162826, k=10)

In [ ]:
content_recommendations = recommend_content_based(
    user_id=162826,
    k=10
)

content_recommendations

In [ ]:
def evaluate_content_based(k=10, max_users=1000):
    users = test_ratings["user_id"].unique()[:max_users]

    precisions = []

    for user_id in users:
        recommendations = recommend_content_based(
            user_id=user_id,
            k=k
        )

        if len(recommendations) == 0:
            continue

        recommended_ids = set(recommendations["id"])

        relevant_ids = set(
            test_ratings[
                (test_ratings["user_id"] == user_id) &
                (test_ratings["rating"] >= 4)
                ]["recipe_id"]
        )

        if len(relevant_ids) == 0:
            continue

        hits = len(recommended_ids & relevant_ids)

        precisions.append(hits / k)

    return np.mean(precisions)

In [ ]:
content_precision_at_10 = evaluate_content_based(
    k=10,
    max_users=1000
)

print(
    "Content-Based Precision@10:",
    content_precision_at_10
)

In [ ]:
recipe_id_to_index = pd.Series(
    recipes.index,
    index=recipes["id"]
)

print("Recipe ID mapping created:", len(recipe_id_to_index))

In [ ]:
def recommend_content_based(user_id, k=10):
    user_ratings = train_ratings[
        train_ratings["user_id"] == user_id
        ].copy()

    # Koristimo samo pozitivne ocene
    user_ratings = user_ratings[
        user_ratings["rating"] >= 4
        ]

    if len(user_ratings) == 0:
        return pd.DataFrame(columns=["id", "name", "score"])

    # Zadržavamo samo recepte koji postoje u recipes tabeli
    user_ratings = user_ratings[
        user_ratings["recipe_id"].isin(recipe_id_to_index.index)
    ]

    if len(user_ratings) == 0:
        return pd.DataFrame(columns=["id", "name", "score"])

    # Indeksi TF-IDF vektora - U ISTOM REDOSLEDU kao ocene
    recipe_indices = user_ratings["recipe_id"].map(
        recipe_id_to_index
    ).values

    user_recipe_vectors = tfidf_matrix[recipe_indices]

    # Ocene su sada pravilno poravnate sa vektorima
    weights = user_ratings["rating"].values.astype(float)

    # User profile
    user_profile = user_recipe_vectors.multiply(
        weights.reshape(-1, 1)
    ).sum(axis=0)

    user_profile = user_profile / weights.sum()
    user_profile = np.asarray(user_profile)

    # Sličnost sa svim receptima
    similarities = cosine_similarity(
        user_profile,
        tfidf_matrix
    ).flatten()

    # Ne preporučujemo već ocenjene recepte
    rated_recipe_ids = user_ratings["recipe_id"].values

    rated_indices = [
        recipe_id_to_index[recipe_id]
        for recipe_id in rated_recipe_ids
    ]

    similarities[rated_indices] = -1

    # Top-k preporuka
    top_indices = np.argsort(similarities)[-k:][::-1]

    recommendations = recipes.iloc[top_indices][
        ["id", "name"]
    ].copy()

    recommendations["score"] = similarities[top_indices]

    return recommendations

In [ ]:
content_recommendations = recommend_content_based(
    user_id=162826,
    k=10
)

content_recommendations

In [ ]:
content_precision_at_10 = evaluate_content_based(
    k=10,
    max_users=1000
)

print(
    "Content-Based Precision@10:",
    content_precision_at_10
)

In [ ]:
def inspect_content_based_hits(max_users=20, k=10):
    users = test_ratings["user_id"].unique()[:max_users]

    results = []

    for user_id in users:
        recommendations = recommend_content_based(
            user_id=user_id,
            k=k
        )

        recommended_ids = set(recommendations["id"])

        relevant_ids = set(
            test_ratings[
                (test_ratings["user_id"] == user_id) &
                (test_ratings["rating"] >= 4)
                ]["recipe_id"]
        )

        hits = recommended_ids & relevant_ids

        results.append({
            "user_id": user_id,
            "relevant_in_test": len(relevant_ids),
            "hits": len(hits),
            "precision": len(hits) / k
        })

    return pd.DataFrame(results)

In [ ]:
content_hit_check = inspect_content_based_hits(
    max_users=20,
    k=10
)

content_hit_check

In [ ]:
user_id = 162826

# Test recepti koje je korisnik dobro ocenio
user_test_relevant = test_ratings[
    (test_ratings["user_id"] == user_id) &
    (test_ratings["rating"] >= 4)
    ].copy()

# Recepti iz treninga koje je korisnik dobro ocenio
user_train_positive = train_ratings[
    (train_ratings["user_id"] == user_id) &
    (train_ratings["rating"] >= 4)
    ].copy()

print("Positive train ratings:", len(user_train_positive))
print("Positive test ratings:", len(user_test_relevant))

In [ ]:
# Indeksi pozitivnih trening recepata
positive_indices = user_train_positive["recipe_id"].map(
    recipe_id_to_index
).dropna().astype(int).values

# Napravi profil samo od pozitivnih recepata
positive_vectors = tfidf_matrix[positive_indices]

positive_profile = positive_vectors.mean(axis=0)
positive_profile = np.asarray(positive_profile)

# Sličnost svih recepata sa pozitivnim profilom
positive_similarities = cosine_similarity(
    positive_profile,
    tfidf_matrix
).flatten()

# Proverimo koliko su test pozitivni recepti slični profilu
test_indices = user_test_relevant["recipe_id"].map(
    recipe_id_to_index
).dropna().astype(int).values

test_similarity_scores = positive_similarities[test_indices]

print("Average similarity of test-positive recipes:",
      test_similarity_scores.mean())

print("Maximum similarity of test-positive recipes:",
      test_similarity_scores.max())

In [ ]:
diagnostic = recipes.iloc[test_indices][
    ["id", "name"]
].copy()

diagnostic["similarity_to_user_profile"] = test_similarity_scores

diagnostic.sort_values(
    "similarity_to_user_profile",
    ascending=False
).head(10)

In [ ]:
def recommend_content_based_max_similarity(user_id, k=10):

    user_ratings = train_ratings[
        (train_ratings["user_id"] == user_id) &
        (train_ratings["rating"] >= 4)
        ].copy()

    if len(user_ratings) == 0:
        return pd.DataFrame(columns=["id", "name", "score"])

    # Samo recepti koji postoje u TF-IDF matrici
    user_ratings = user_ratings[
        user_ratings["recipe_id"].isin(recipe_id_to_index.index)
    ]

    if len(user_ratings) == 0:
        return pd.DataFrame(columns=["id", "name", "score"])

    # Indeksi pozitivno ocenjenih recepata
    positive_indices = user_ratings["recipe_id"].map(
        recipe_id_to_index
    ).values

    positive_vectors = tfidf_matrix[positive_indices]

    # Računamo sličnost u batch-evima da ne zauzmemo previše RAM-a
    batch_size = 10000
    max_scores = np.full(tfidf_matrix.shape[0], -1.0)

    for start in range(0, tfidf_matrix.shape[0], batch_size):
        end = min(start + batch_size, tfidf_matrix.shape[0])

        batch = tfidf_matrix[start:end]

        similarities = cosine_similarity(
            batch,
            positive_vectors
        )

        max_scores[start:end] = similarities.max(axis=1)

    # Ne preporučujemo već ocenjene recepte
    rated_recipe_ids = user_ratings["recipe_id"].values

    rated_indices = [
        recipe_id_to_index[recipe_id]
        for recipe_id in rated_recipe_ids
    ]

    max_scores[rated_indices] = -1

    # Top-k
    top_indices = np.argsort(max_scores)[-k:][::-1]

    recommendations = recipes.iloc[top_indices][
        ["id", "name"]
    ].copy()

    recommendations["score"] = max_scores[top_indices]

    return recommendations

In [ ]:
content_max_recommendations = recommend_content_based_max_similarity(
    user_id=162826,
    k=10
)

content_max_recommendations

In [ ]:
def evaluate_content_based_max_similarity(k=10, max_users=1000):
    users = test_ratings["user_id"].unique()[:max_users]

    precisions = []

    for user_id in users:

        recommendations = recommend_content_based_max_similarity(
            user_id=user_id,
            k=k
        )

        if len(recommendations) == 0:
            continue

        recommended_ids = set(recommendations["id"])

        relevant_ids = set(
            test_ratings[
                (test_ratings["user_id"] == user_id) &
                (test_ratings["rating"] >= 4)
                ]["recipe_id"]
        )

        if len(relevant_ids) == 0:
            continue

        hits = len(recommended_ids & relevant_ids)

        precisions.append(hits / k)

    return np.mean(precisions)

In [ ]:
content_max_precision_at_10 = evaluate_content_based_max_similarity(
    k=10,
    max_users=1000
)

print(
    "Content-Based Max Similarity Precision@10:",
    content_max_precision_at_10
)

In [ ]:
content_max_precision_at_10 = evaluate_content_based_max_similarity(
    k=10,
    max_users=100
)

print(
    "Content-Based Max Similarity Precision@10:",
    content_max_precision_at_10
)

In [ ]:
def evaluate_content_based_fast(k=10, max_users=1000, batch_size=10):
    users = test_ratings["user_id"].unique()[:max_users]

    precisions = []

    for start in range(0, len(users), batch_size):
        batch_users = users[start:start + batch_size]

        profiles = []
        valid_users = []

        for user_id in batch_users:

            user_ratings = train_ratings[
                (train_ratings["user_id"] == user_id) &
                (train_ratings["rating"] >= 4)
                ]

            user_ratings = user_ratings[
                user_ratings["recipe_id"].isin(
                    recipe_id_to_index.index
                )
            ]

            if len(user_ratings) == 0:
                continue

            indices = user_ratings["recipe_id"].map(
                recipe_id_to_index
            ).values

            vectors = tfidf_matrix[indices]

            weights = user_ratings["rating"].values.astype(float)

            profile = vectors.multiply(
                weights.reshape(-1, 1)
            ).sum(axis=0)

            profile = profile / weights.sum()

            profiles.append(profile)
            valid_users.append(user_id)

        if len(profiles) == 0:
            continue

        profiles = np.vstack([
            np.asarray(profile).ravel()
            for profile in profiles
        ])

        # Sličnost svih profila iz batch-a sa svim receptima
        similarities = cosine_similarity(
            profiles,
            tfidf_matrix
        )

        for i, user_id in enumerate(valid_users):

            scores = similarities[i].copy()

            # Ne preporučujemo recepte koje je korisnik već ocenio
            train_user_recipe_ids = train_ratings[
                train_ratings["user_id"] == user_id
                ]["recipe_id"].values

            rated_indices = [
                recipe_id_to_index[recipe_id]
                for recipe_id in train_user_recipe_ids
                if recipe_id in recipe_id_to_index.index
            ]

            scores[rated_indices] = -1

            top_indices = np.argpartition(
                scores,
                -k
            )[-k:]

            recommended_ids = set(
                recipes.iloc[top_indices]["id"]
            )

            relevant_ids = set(
                test_ratings[
                    (test_ratings["user_id"] == user_id) &
                    (test_ratings["rating"] >= 4)
                    ]["recipe_id"]
            )

            if len(relevant_ids) == 0:
                continue

            hits = len(
                recommended_ids & relevant_ids
            )

            precisions.append(hits / k)

        print(
            f"Processed {min(start + batch_size, len(users))}/{len(users)} users"
        )

    return np.mean(precisions)

In [ ]:
content_precision_at_10_fast = evaluate_content_based_fast(
    k=10,
    max_users=100,
    batch_size=10
)

print(
    "Content-Based Precision@10:",
    content_precision_at_10_fast
)

In [ ]:
def evaluate_content_based_recall_fast(k=10, max_users=100, batch_size=10):
    users = test_ratings["user_id"].unique()[:max_users]

    recalls = []

    for start in range(0, len(users), batch_size):
        batch_users = users[start:start + batch_size]

        profiles = []
        valid_users = []

        for user_id in batch_users:

            user_ratings = train_ratings[
                (train_ratings["user_id"] == user_id) &
                (train_ratings["rating"] >= 4)
                ]

            user_ratings = user_ratings[
                user_ratings["recipe_id"].isin(
                    recipe_id_to_index.index
                )
            ]

            if len(user_ratings) == 0:
                continue

            indices = user_ratings["recipe_id"].map(
                recipe_id_to_index
            ).values

            vectors = tfidf_matrix[indices]

            weights = user_ratings["rating"].values.astype(float)

            profile = vectors.multiply(
                weights.reshape(-1, 1)
            ).sum(axis=0)

            profile = profile / weights.sum()

            profiles.append(profile)
            valid_users.append(user_id)

        if len(profiles) == 0:
            continue

        profiles = np.vstack([
            np.asarray(profile).ravel()
            for profile in profiles
        ])

        similarities = cosine_similarity(
            profiles,
            tfidf_matrix
        )

        for i, user_id in enumerate(valid_users):

            scores = similarities[i].copy()

            train_user_recipe_ids = train_ratings[
                train_ratings["user_id"] == user_id
                ]["recipe_id"].values

            rated_indices = [
                recipe_id_to_index[recipe_id]
                for recipe_id in train_user_recipe_ids
                if recipe_id in recipe_id_to_index.index
            ]

            scores[rated_indices] = -1

            top_indices = np.argpartition(
                scores,
                -k
            )[-k:]

            recommended_ids = set(
                recipes.iloc[top_indices]["id"]
            )

            relevant_ids = set(
                test_ratings[
                    (test_ratings["user_id"] == user_id) &
                    (test_ratings["rating"] >= 4)
                    ]["recipe_id"]
            )

            if len(relevant_ids) == 0:
                continue

            hits = len(
                recommended_ids & relevant_ids
            )

            recalls.append(
                hits / len(relevant_ids)
            )

        print(
            f"Processed {min(start + batch_size, len(users))}/{len(users)} users"
        )

    return np.mean(recalls)

In [ ]:
content_recall_at_10 = evaluate_content_based_recall_fast(
    k=10,
    max_users=100,
    batch_size=10
)

print(
    "Content-Based Recall@10:",
    content_recall_at_10
)